# Exploratory Data Analysis — Retail Price Optimization Dataset

This notebook walks through the dataset used by the Dynamic Pricing Engine.
We will:
1. Load and inspect the data
2. Check for missing values
3. Visualise distributions and correlations
4. Explore relationships between price, demand, and competitor pricing

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline

# Load the dataset
DATA_PATH = Path("../data/retail_price_dataset.csv")
df = pd.read_csv(DATA_PATH)
print(f"Shape: {df.shape}")
df.head()

## 1. Basic Info & Missing Values

In [ ]:
# Data types and non-null counts
df.info()

In [ ]:
# Missing value heatmap
plt.figure(figsize=(12, 4))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap="viridis")
plt.title("Missing Values Heatmap")
plt.tight_layout()
plt.show()

print("\nMissing counts:")
print(df.isnull().sum()[df.isnull().sum() > 0])

## 2. Descriptive Statistics

In [ ]:
df.describe().T

## 3. Distribution of Key Numerical Features

In [ ]:
num_cols = ["unit_price", "qty", "freight_price", "comp_1", "product_score", "customers"]
existing = [c for c in num_cols if c in df.columns]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, col in zip(axes.flat, existing):
    df[col].hist(bins=40, ax=ax, edgecolor="white")
    ax.set_title(col)
plt.suptitle("Feature Distributions", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 4. Correlation Matrix

In [ ]:
plt.figure(figsize=(14, 10))
corr = df.select_dtypes(include=[np.number]).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="coolwarm",
            linewidths=0.5, square=True)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 5. Price vs Demand Relationship

In [ ]:
if "unit_price" in df.columns and "qty" in df.columns:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].scatter(df["unit_price"], df["qty"], alpha=0.3, s=10)
    axes[0].set_xlabel("Unit Price")
    axes[0].set_ylabel("Quantity (Demand)")
    axes[0].set_title("Price vs Demand")

    if "comp_1" in df.columns:
        df["price_diff_eda"] = df["unit_price"] - df["comp_1"]
        axes[1].scatter(df["price_diff_eda"], df["qty"], alpha=0.3, s=10, color="coral")
        axes[1].set_xlabel("Price Difference (ours − competitor)")
        axes[1].set_ylabel("Quantity (Demand)")
        axes[1].set_title("Price Gap vs Demand")

    plt.tight_layout()
    plt.show()

## 6. Category-Level Analysis

In [ ]:
if "product_category_name" in df.columns:
    top_cats = df["product_category_name"].value_counts().head(10)
    plt.figure(figsize=(10, 5))
    sns.barplot(x=top_cats.values, y=top_cats.index, palette="viridis")
    plt.xlabel("Count")
    plt.title("Top 10 Product Categories")
    plt.tight_layout()
    plt.show()

## 7. Box Plots — Price by Category

In [ ]:
if "product_category_name" in df.columns and "unit_price" in df.columns:
    top10 = df["product_category_name"].value_counts().head(10).index
    subset = df[df["product_category_name"].isin(top10)]
    plt.figure(figsize=(12, 6))
    sns.boxplot(data=subset, x="product_category_name", y="unit_price")
    plt.xticks(rotation=45, ha="right")
    plt.title("Unit Price Distribution by Category")
    plt.tight_layout()
    plt.show()

## Summary

Key observations from this EDA:
- Examine the correlation matrix to identify strongly predictive features.
- The price–demand scatter plot shows how pricing affects quantity sold.
- Competitor price differences may be a strong signal for the model.
- Category-level analysis reveals which segments have the most data.

These insights inform feature engineering choices in `src/feature_engineering.py`.